# W4 Homework — Follow the Evidence in a Sales Investigation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week04/W4_hw_sales_investigation.ipynb)

**Goal:** complete the agent loop, use an observed result to select a follow-up question, and explain why the next action or final answer follows from the evidence.

Use the same synthetic shop, tools, and ReAct instructions as the lab. This homework compares **January and February 2025**; it does not reuse the lab's February–March answers. All sales are in USD and include only completed orders. Define **change = February sales − January sales**.

1. Complete the loop's execute → save → stop block.
2. Compare categories, then select the category with the largest increase and investigate its products.
3. Record your findings and explain one Observation → next decision transition.
4. Run the format check and submit the executed notebook on the LMS. Due date: see the LMS.

The student check verifies **format only**: fields, types, and the structure of saved records. It does not compare your numbers with correct answers or award a correctness score. Loop behavior, findings, and explanations are graded after submission.

Allow about 60–75 minutes. Complete the marked `FILL IN` blocks before running the cells that use them. You may refer to the lab.

## 1. Setup and identity

In Colab choose **File → Save a copy in Drive**, then run these cells. The helper downloads automatically. Enter your API key in the hidden input when prompted; do not paste it into notebook code. Local Jupyter can use an existing `OPENAI_API_KEY` environment variable.

The helper supplies data generation, arithmetic, request descriptions, and trace display. The model call and loop remain visible below. This is the same seeded teaching database as the lab, not real customer transactions.

In [ ]:
%pip install -q "openai==1.109.1"

In [ ]:
import json
import os
import sqlite3
from getpass import getpass
from pathlib import Path
from urllib.request import urlopen

# Download the supplied data/tool utilities when opening in a fresh Colab session.
SUPPORT_URL = "https://raw.githubusercontent.com/ralbu85/stml_2026/main/lectures/week04/w4_support.py"
if not Path("w4_support.py").is_file():
    with urlopen(SUPPORT_URL, timeout=30) as response:
        Path("w4_support.py").write_bytes(response.read())

from openai import OpenAI
from w4_support import prepare_shop, calculate, finish, action_tools, show_exchange, show_trace, DB_CONTEXT

API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key (input hidden): ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4.1"
MAX_CALLS = 10
DB_PATH = prepare_shop(Path("outputs/w4_homework/shop.sqlite"))
print("Ready:", DB_PATH)

In [ ]:
### FILL IN (START) ###
STUDENT_NAME = ""
STUDENT_ID = ""
### FILL IN (END) ###
print("submitting as:", STUDENT_NAME or "?", STUDENT_ID or "?")

### Provided tool and model instructions

`run_sql(query)` returns column names, rows, and a row count, or an error. `calculate` accepts numeric arithmetic expressions; `finish` submits an answer without grading it. SQL can compute the requested values directly. A single sufficient query is valid.

`orders` supplies `ordered_at` and `status`. `order_items` supplies `quantity` and `unit_price`. `products` supplies each product's `name` and `category`. Join on `order_id` and `product_id` as in the lab. Use `quantity * unit_price` for sales.

In [ ]:
def run_sql(query: str) -> dict:
    """Execute one read-only SQLite query. Return all rows and row_count, or an error if the result exceeds 1,000 rows.

    Args:
        query: One read-only SQLite query using the supplied schema.
    """
    try:
        with sqlite3.connect(f'file:{DB_PATH.resolve()}?mode=ro', uri=True) as db:
            db.execute('PRAGMA query_only = ON')
            cursor = db.execute(query)
            rows = cursor.fetchmany(1001)
            if len(rows) > 1000:
                return {'error': 'Result exceeds 1,000 rows. No partial result was returned. Submit a query with a smaller result.'}
            return {'columns': [c[0] for c in cursor.description or []],
                    'rows': rows, 'row_count': len(rows)}
    except sqlite3.Error as error:
        return {'error': str(error)}

In [ ]:
BASE = """Answer the user's question using this shop database. Respond in English.
Sales are quantity * unit_price for completed orders; exclude cancelled orders.
Use SQL to filter, join, and aggregate database rows. Count the requested unit:
an order may have several item rows. Use calculate for arithmetic on returned
numbers when needed; SQL may also compute the final value directly.
Consult the data before answering. Reuse returned evidence; correct failed queries.
Submit the supported answer with finish, stating any unresolved limitation.
A successful query or finish does not prove that an answer is correct.
""" + DB_CONTEXT
BASIC_TOOLS = action_tools()
operations = {"run_sql": run_sql, "calculate": calculate, "finish": finish}
REACT = """
Before each action, write thought: assess the evidence available so far,
what is still missing, and why this next action addresses it.
After an observation, check what the query or calculation actually measured.
If it was wrong, revise the next action. Before finish, explain how the
observations support each requested part of the answer or leave a limitation.
"""
REACT_TOOLS = action_tools(react=True)

## 2. Complete execute → save → stop

The request step A is provided. Complete B–D inside `run_loop`:

- Call the Python operation named by the model with its supplied input.
- Append the **full assistant message** and the **actual tool result** to `messages`. The tool message must use the same `call.id` so that the next model call can use the Observation.
- Display the exchange using `show_exchange(request, result)`.
- Stop only when `finish` returns an `answer`. Return `{"answer": ..., "messages": messages, "calls": turn + 1}`.

An error is also an Observation: save it and allow the next request to respond to it. At the call limit, the provided final lines return `answer=None`. Do not add model calls outside the loop or change the supplied database.

In [ ]:
def run_loop(question, instructions=BASE, tools=BASIC_TOOLS, max_calls=MAX_CALLS):
    messages = [{"role": "system", "content": instructions},
                {"role": "user", "content": question}]
    for turn in range(max_calls):
        # A. Ask for the next request using all messages so far.
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice="required", parallel_tool_calls=False,
            temperature=0, max_tokens=2400,
        )
        message = response.choices[0].message
        if len(message.tool_calls or []) != 1:
            raise RuntimeError("Expected one tool request; inspect response.")
        call = message.tool_calls[0]
        request = json.loads(call.function.arguments)
        # Complete B–D using the same contract as the lab.
        ### FILL IN (START) ###
        # B. Execute request["action"] with request["action_input"].
        # C. Save the assistant message and the tool result, including call.id.
        #    Use json.dumps(result) for the tool message content.
        # D. Return answer/messages/calls only after a successful finish.
        raise NotImplementedError("Complete execution, message saving, and stopping.")
        ### FILL IN (END) ###
    print("Call limit reached without a submitted answer.")
    return {"answer": None, "messages": messages, "calls": max_calls}

## 3. Compare categories, then choose a follow-up

### 3.1 Run the category comparison

Run the question below with your completed loop. Read the SQL filters, returned values, and the next Thought. Check that the months are January and February 2025 and that cancelled orders are excluded.

If `answer` is `None`, inspect the saved messages and repair the missing loop behavior or clarify the question before rerunning. A completed run can still contain incorrect SQL or an unsupported answer.

In [ ]:
category_question = (
    "Compare completed sales in January and February 2025 by category. "
    "Show each month's sales and the change (February minus January), "
    "plus the overall change. Which category has the largest increase?"
)
category_run = run_loop(category_question, instructions=BASE + REACT, tools=REACT_TOOLS)
print("Category answer:", category_run["answer"])

### 3.2 Select a category from that result

Set `CATEGORY_TO_INSPECT` to the category with the largest increase. Do not copy the lab's choice without checking this period's result.

Each `run_loop` call starts a **fresh conversation**. The new question must explicitly include the selected category and dates. The question below does that for you. It also asks for every product, including zero sales in a month. Its SQL hint separates eligible sales from the full product list.

In [ ]:
### FILL IN (START) ###
CATEGORY_TO_INSPECT = ""
### FILL IN (END) ###
if not CATEGORY_TO_INSPECT.strip():
    raise ValueError("Select the category from the preceding result first.")
product_question = (
    f"For {CATEGORY_TO_INSPECT}, compare completed sales by product in January "
    "and February 2025. Show each month's sales and the change (February minus January). "
    "Keep every product in this category, showing zero for a month with no completed sales. "
    "Aggregate completed order items by product and month first, then join those totals "
    "to the full product list and fill missing totals with zero. "
    "Which products account for the category's change? "
    "What can these sales records establish, and what remains unknown about why sales changed?"
)
product_run = run_loop(product_question, instructions=BASE + REACT, tools=REACT_TOOLS)
print("Product answer:", product_run["answer"])

### Read the saved exchanges

The next cell makes no API calls. Use the saved SQL and actual Observations to check the scope, missing products, and whether product changes add to the category change. You may use `run_sql` for additional independent queries in a new cell above the final format check; record corrections in your explanation.

In [ ]:
show_trace(category_run["messages"])
show_trace(product_run["messages"])

## 4. Record findings and explain a decision

Enter the findings as **literal lists of dictionaries**, using numeric USD amounts rather than formatted strings. Include every category and every product in the selected category. Do not paste a model's formatted table as one string.

- `category_findings`: each row has `category` (text), `jan_sales`, `feb_sales`, and `change` (numbers).
- `product_findings`: each row has `product` (text), `jan_sales`, `feb_sales`, and `change` (numbers).

Use the actual returned evidence. If the model's answer conflicts with the data, inspect or query further, enter the corrected findings, and explain the correction. No expected answers are included in this notebook.

In [ ]:
### FILL IN (START) ###
category_findings = []
product_findings = []
### FILL IN (END) ###

### Explain one transition in your own words

Choose a non-final Observation and the next request from one saved run. `run` is `"categories"` or `"products"`; `observation_exchange` is its **1-based exchange number** in the trace.

In `observation`, identify the returned value or error you used. In `next_decision`, describe the following Thought and Action and explain whether they were justified. In `category_choice`, explain the numerical reason for selecting your category and why the follow-up needed to state it again. In `limitation`, distinguish the measured change from a cause that would require other data. Use `correction` for any discrepancy you repaired, or say that you found none and state what you checked.

In [ ]:
### FILL IN (START) ###
explanation = {
    "run": "",
    "observation_exchange": None,
    "observation": "",
    "next_decision": "",
    "category_choice": "",
    "limitation": "",
    "correction": "",
}
### FILL IN (END) ###

## 5. Format check and submission

This check tests **only submission format**. `PASS` and `HOMEWORK COMPLETE` mean the required fields and records are present; they do **not** mean the answer or loop is correct. Correctness is assessed after submission by the instructor.

Run the notebook from top to bottom after completing your work. Keep the model/tool outputs, run this check last, then use **File → Download → Download .ipynb** and upload the file to the LMS by its listed deadline. Submit the notebook itself, not screenshots. The hidden API input is not saved in the notebook.

In [ ]:
import math

def text_filled(value):
    return isinstance(value, str) and bool(value.strip())

def table_format(rows, label):
    fields = {label, "jan_sales", "feb_sales", "change"}
    return isinstance(rows, list) and bool(rows) and all(
        isinstance(row, dict) and set(row) == fields and text_filled(row[label])
        and all(type(row[key]) in (int, float) and math.isfinite(row[key])
                for key in ("jan_sales", "feb_sales", "change"))
        for row in rows
    )

def run_format(run):
    if not isinstance(run, dict) or not {"answer", "messages", "calls"} <= run.keys():
        return False
    messages = run["messages"]
    return (text_filled(run["answer"]) and type(run["calls"]) is int
            and isinstance(messages, list) and all(isinstance(m, dict) for m in messages)
            and any(m.get("role") == "assistant" and m.get("tool_calls") for m in messages)
            and any(m.get("role") == "tool" and text_filled(m.get("content")) for m in messages))

entry = globals().get("explanation", {})
if not isinstance(entry, dict):
    entry = {}
selected = {"categories": globals().get("category_run"), "products": globals().get("product_run")}.get(entry.get("run"))
messages = selected.get("messages", []) if isinstance(selected, dict) else []
if not isinstance(messages, list):
    messages = []
observations = [m for m in messages if isinstance(m, dict) and m.get("role") == "tool"]
index = entry.get("observation_exchange")
completion = {
    "name and student ID": text_filled(globals().get("STUDENT_NAME")) and text_filled(globals().get("STUDENT_ID")),
    "loop function defined": callable(globals().get("run_loop")),
    "category run: answer and saved record fields": run_format(globals().get("category_run")),
    "product run: answer and saved record fields": run_format(globals().get("product_run")),
    "selected category: text": text_filled(globals().get("CATEGORY_TO_INSPECT")),
    "category findings: column names and types": table_format(globals().get("category_findings"), "category"),
    "product findings: column names and types": table_format(globals().get("product_findings"), "product"),
    "explanation: run and a non-final exchange number": type(index) is int and 1 <= index < len(observations),
    "explanation: all text fields filled": all(text_filled(entry.get(k)) for k in
        ("observation", "next_decision", "category_choice", "limitation", "correction")),
}
print("submitted by:", globals().get("STUDENT_NAME") or "?", globals().get("STUDENT_ID") or "?", "\n")
for label, ok in completion.items():
    print("PASS " if ok else "FAIL ", label)
print("\nHOMEWORK COMPLETE — format only; download and submit" if all(completion.values()) else "\nNOT COMPLETE YET")